# 🧠 Da7ee7-El-Dof3a — Kaggle AI Service

This notebook is the full **AI Service** for Da7ee7-El-Dof3a (دحيح الدفعة).

It runs on a Kaggle GPU instance and exposes a FastAPI server through an
**ngrok** tunnel, which the local backend calls for all heavy AI work:

1. Install dependencies & detect GPU
2. Compare 5 candidate LLMs and auto-select the best one
3. Apply BitsAndBytes quantization (4-bit NF4, falling back to 8-bit)
4. Load documents (PDF / PPTX / DOCX), transcribe audio & video with Whisper
5. Chunk text, build embeddings, index with FAISS
6. Build a LangChain RetrievalQA pipeline
7. Generate Smart Summary, Important Topics, Solved Exams, Revision Notes
8. Export results as PDF
9. Expose everything via FastAPI + ngrok

> **Before running:** In Kaggle → *Add-ons → Secrets*, add a secret named
> `NGROK_AUTH_TOKEN` with your ngrok auth token. Also enable **GPU T4 x2**
> (or better) under *Settings → Accelerator*.

In [ ]:
!pip show numpy
!pip show torch
!pip show transformers

In [ ]:
!pip install -q \
langchain==0.2.16 \
langchain-community==0.2.16 \
langchain-core==0.2.38 \
langchain-text-splitters==0.2.4 \
langchain-huggingface==0.0.3

In [ ]:
!pip install -q python-pptx python-docx pypdf

In [ ]:
!pip install -q openai-whisper

In [ ]:
!pip install -q reportlab

In [ ]:
!pip install -q pyngrok

In [ ]:
!pip install -U bitsandbytes accelerate

In [ ]:
!pip install -q faiss-cpu

In [2]:
# =============================================================================
# 2. IMPORT LIBRARIES
# =============================================================================

import os
import gc
import re
import json
import time
import shutil
import logging
import threading
import tempfile
from pathlib import Path
from typing import Optional, Any
from xml.sax.saxutils import escape as xml_escape

import torch
import numpy as np

# ==========================
# Hugging Face
# ==========================
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline as hf_pipeline,
)

# ==========================
# LangChain
# ==========================
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

from langchain_community.vectorstores import FAISS

from langchain_huggingface import (
    HuggingFaceEmbeddings,
    HuggingFacePipeline,
)
# NOTE: `langchain.chains.RetrievalQA` was imported in the original notebook but
# never actually used (summaries were generated manually via run_llm()). Removed
# as dead code — the summarization pipeline below is a hand-rolled Map-Reduce
# chain instead, which gives us full control over chunk batching and logging.

# ==========================
# Document Processing
# ==========================
from pypdf import PdfReader
from pptx import Presentation
from docx import Document as DocxDocument

# ==========================
# Audio / Video
# ==========================
import whisper
try:
    from moviepy import VideoFileClip          # moviepy >= 2.0
except ImportError:
    from moviepy.editor import VideoFileClip   # moviepy < 2.0 fallback

# ==========================
# PDF Export
# ==========================
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    ListFlowable,
    ListItem,
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# ==========================
# FastAPI
# ==========================
from fastapi import FastAPI, UploadFile, File, Form, Request
from fastapi.responses import FileResponse, JSONResponse

import uvicorn
import nest_asyncio

# ==========================
# ngrok
# ==========================
from pyngrok import ngrok, conf as ngrok_conf

nest_asyncio.apply()

# =============================================================================
# LOGGING — every stage below (ingest, indexing, retrieval, generation, PDF
# export, downloads) logs through this logger so failures are traceable
# instead of silently returning bad output.
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("da7ee7_ai_service")
logger.info("All libraries imported successfully.")


2026-07-26 14:12:54 | INFO     | da7ee7_ai_service | All libraries imported successfully.


In [3]:
# 3. DETECT GPU
# -----------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    logger.info(f"GPU detected: {gpu_name} ({total_mem_gb:.1f} GB)")
else:
    logger.warning("No GPU detected. Model comparison / inference will be very slow.")

WORKDIR = Path("/kaggle/working/da7ee7")
UPLOADS_DIR = WORKDIR / "uploads"
INDEX_DIR = WORKDIR / "faiss_indexes"
OUTPUT_DIR = WORKDIR / "outputs"
for d in (UPLOADS_DIR, INDEX_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

logger.info(f"Working directories ready under {WORKDIR}")


2026-07-26 14:12:54 | INFO     | da7ee7_ai_service | GPU detected: Tesla T4 (14.6 GB)
2026-07-26 14:12:54 | INFO     | da7ee7_ai_service | Working directories ready under /kaggle/working/da7ee7


## 4. Model comparison — pick the best quantized LLM for this session

In [4]:
# 4a. Candidate models to compare.
# NOTE: some of these require an accepted license / gated-repo access on
# Hugging Face (e.g. Llama, Gemma). Log in with `huggingface-cli login` or
# set the HF_TOKEN Kaggle secret before running this cell if needed.
# -----------------------------------------------------------------------------
CANDIDATE_MODELS = [
    {"name": "Qwen/Qwen2.5-7B-Instruct",        "max_ctx": 32768},
    {"name": "google/gemma-3-4b-it",             "max_ctx": 8192},
    {"name": "meta-llama/Llama-3.2-3B-Instruct", "max_ctx": 8192},
    {"name": "microsoft/Phi-4-mini-instruct",    "max_ctx": 16384},
    {"name": "mistralai/Mistral-7B-Instruct-v0.3","max_ctx": 32768},
]

BENCHMARK_PROMPTS = [
    "Summarize in two sentences: Newton's second law states that force equals mass times acceleration.",
    "What is the time complexity of binary search on a sorted array of size n?",
    "Explain the difference between supervised and unsupervised learning in one paragraph.",
]

In [5]:
# 4b. Quantization config — prefer 4-bit NF4, fall back to 8-bit if unsupported.
# -----------------------------------------------------------------------------
def build_quant_config() -> tuple[BitsAndBytesConfig, str]:
    try:
        cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        return cfg, "4-bit NF4"
    except Exception as exc:
        logger.warning(f"4-bit NF4 unsupported ({exc}), falling back to 8-bit.")
        cfg = BitsAndBytesConfig(load_in_8bit=True)
        return cfg, "8-bit"


QUANT_CONFIG, QUANT_MODE = build_quant_config()
logger.info(f"Using quantization mode: {QUANT_MODE}")


2026-07-26 14:12:54 | INFO     | da7ee7_ai_service | Using quantization mode: 4-bit NF4


In [6]:
# 4c. Load + benchmark each candidate model: accuracy proxy, speed, memory, context length.
# -----------------------------------------------------------------------------
def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # FIX: several candidate models (Qwen, Llama, Mistral) ship WITHOUT a pad
    # token. Leaving this unset causes noisy "Setting pad_token_id to
    # eos_token_id" warnings on every single generation call, and can cause
    # unpredictable batching/attention-mask behavior. We set it once, here,
    # for every model — not per-call.
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=QUANT_CONFIG,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return model, tokenizer


def score_response_quality(prompt: str, response: str) -> float:
    """Cheap heuristic proxy for 'accuracy': response length, relevance
    keyword overlap, and absence of degenerate repetition. Replace with a
    held-out labeled eval set for a rigorous benchmark."""
    if not response.strip():
        return 0.0
    words = response.split()
    unique_ratio = len(set(words)) / max(len(words), 1)
    length_score = min(len(words) / 60, 1.0)
    return round(0.6 * unique_ratio + 0.4 * length_score, 3)


def benchmark_model(model_name: str, max_ctx: int) -> Optional[dict]:
    logger.info(f"--- Benchmarking {model_name} ---")
    try:
        model, tokenizer = load_model_and_tokenizer(model_name)
    except Exception as exc:
        logger.warning(f"Skipping {model_name}: failed to load ({exc})")
        return None

    # Use the SAME deterministic decoding strategy we'll use in production
    # (greedy + repetition penalty) so the benchmark scores are representative
    # of real output quality, not sampling-noise output quality.
    gen = hf_pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        return_full_text=False,
        max_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.25,
        no_repeat_ngram_size=4,
    )

    scores, latencies = [], []
    for prompt in BENCHMARK_PROMPTS:
        start = time.perf_counter()
        out = gen(prompt)[0]["generated_text"]
        latencies.append(time.perf_counter() - start)
        scores.append(score_response_quality(prompt, out))

    mem_gb = torch.cuda.memory_allocated() / (1024 ** 3) if DEVICE == "cuda" else 0.0

    result = {
        "model": model_name,
        "accuracy_proxy": round(float(np.mean(scores)), 3),
        "avg_latency_sec": round(float(np.mean(latencies)), 2),
        "memory_gb": round(mem_gb, 2),
        "context_length": max_ctx,
    }

    # Free GPU memory before loading the next candidate.
    del model, tokenizer, gen
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    logger.info(result)
    return result


In [7]:
# 4d. Run the comparison and select the best model.
# Scoring: normalized weighted sum of accuracy (higher better), speed (lower
# latency better), memory footprint (lower better), context length (higher better).
# -----------------------------------------------------------------------------
benchmark_results = []
for candidate in CANDIDATE_MODELS:
    result = benchmark_model(candidate["name"], candidate["max_ctx"])
    if result:
        benchmark_results.append(result)

if not benchmark_results:
    raise RuntimeError(
        "No candidate model could be loaded. Check Hugging Face access/licensing "
        "for gated models (Llama, Gemma) and your Kaggle GPU quota."
    )


def normalize(values, higher_is_better=True):
    lo, hi = min(values), max(values)
    if hi == lo:
        return [1.0 for _ in values]
    return [
        (v - lo) / (hi - lo) if higher_is_better else (hi - v) / (hi - lo)
        for v in values
    ]


accs = normalize([r["accuracy_proxy"] for r in benchmark_results], True)
speeds = normalize([r["avg_latency_sec"] for r in benchmark_results], False)
mems = normalize([r["memory_gb"] for r in benchmark_results], False)
ctxs = normalize([r["context_length"] for r in benchmark_results], True)

WEIGHTS = {"accuracy": 0.4, "speed": 0.25, "memory": 0.2, "context": 0.15}

for i, r in enumerate(benchmark_results):
    r["final_score"] = round(
        WEIGHTS["accuracy"] * accs[i]
        + WEIGHTS["speed"] * speeds[i]
        + WEIGHTS["memory"] * mems[i]
        + WEIGHTS["context"] * ctxs[i],
        4,
    )

benchmark_results.sort(key=lambda r: r["final_score"], reverse=True)
BEST_MODEL = benchmark_results[0]

logger.info("=== Model comparison results (best first) ===")
for r in benchmark_results:
    logger.info(r)

logger.info(f"Selected model: {BEST_MODEL['model']} (score={BEST_MODEL['final_score']})")


2026-07-26 14:12:54 | INFO     | da7ee7_ai_service | --- Benchmarking Qwen/Qwen2.5-7B-Instruct ---
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:12:54 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/vocab.json "HTTP/1.1 200 OK"
2026-07-26 14:12:54 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/vocab.json "HTTP/1.1 200 OK"


...

2026-07-26 14:16:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3/resolve/c170c708c41dac9275d15a8fff4eca08d52bab71/model-00001-of-00003.safetensors "HTTP/1.1 302 Found"
2026-07-26 14:16:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3/resolve/c170c708c41dac9275d15a8fff4eca08d52bab71/model-00003-of-00003.safetensors "HTTP/1.1 302 Found"
2026-07-26 14:16:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3/resolve/c170c708c41dac9275d15a8fff4eca08d52bab71/model-00002-of-00003.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-07-26 14:18:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:18:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.3/c170c708c41dac9275d15a8fff4eca08d52bab71/generation_config.json "HTTP/1.1 200 OK"
2026-07-26 14:18:16 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.3/c170c708c41dac9275d15a8fff4eca08d52bab71/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-07-26 14:18:43 | INFO     | da7ee7_ai_service | {'model': 'mistralai/Mistral-7B-Instruct-v0.3', 'accuracy_proxy': 0.875, 'avg_latency_sec': 8.9, 'memory_gb': 1.62, 'context_length': 32768}
2026-07-26 14:18:43 | INFO     | da7ee7_ai_service | === Mo

In [8]:
# 4e. Load the winning model for real use (kept in memory for the rest of the session).
# -----------------------------------------------------------------------------
SELECTED_MODEL_NAME = BEST_MODEL["model"]
llm_model, llm_tokenizer = load_model_and_tokenizer(SELECTED_MODEL_NAME)

# -----------------------------------------------------------------------------
# GENERATION CONFIG — this is the main fix for hallucinated / repeated /
# looping output:
#   - do_sample=False, num_beams=1  -> greedy decoding: fully deterministic,
#     removes the random-sampling noise that caused made-up facts.
#   - repetition_penalty=1.25       -> penalizes tokens already generated,
#     directly targets "repeated words" / "repeated sentences".
#   - no_repeat_ngram_size=4        -> hard-blocks any 4-gram from repeating
#     verbatim, which is what stops the endless-loop failure mode outright.
#   - max_new_tokens=300 (within the 250-350 sweet spot) + min_new_tokens=16
#     -> bounded output length, avoids run-on generation.
#   - eos_token_id / pad_token_id set explicitly -> removes the "pad token
#     not set" warning and lets the model stop cleanly at its own EOS.
#   - return_full_text=False        -> the pipeline returns ONLY the newly
#     generated text. The original code manually tried to strip the prompt
#     back out with `output[len(prompt_text):]`, which silently failed
#     whenever the model's chat template altered the text (adding special
#     tokens), causing the ENTIRE prompt — instructions included — to leak
#     into the "summary". That prompt-echo was a major source of the
#     reported hallucination/garbage-output symptom.
#
# NOTE: temperature / top_p are intentionally NOT passed here. They only
# apply when do_sample=True; passing them anyway (as the original code did
# with do_sample=True, temperature=0.3) is what reintroduced sampling
# randomness, and passing them with do_sample=False instead just produces a
# UserWarning for no benefit. Omitting them is the correct fix, not just
# setting temperature=0 (which is invalid for HF's sampler).
# -----------------------------------------------------------------------------
GENERATION_KWARGS = dict(
    max_new_tokens=300,
    min_new_tokens=16,
    do_sample=False,
    num_beams=1,
    repetition_penalty=1.25,
    no_repeat_ngram_size=4,
)

text_gen_pipeline = hf_pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    eos_token_id=llm_tokenizer.eos_token_id,
    pad_token_id=llm_tokenizer.pad_token_id,
    return_full_text=False,
    **GENERATION_KWARGS,
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
logger.info(f"LLM ready: {SELECTED_MODEL_NAME} | quantization: {QUANT_MODE} | generation: {GENERATION_KWARGS}")


2026-07-26 14:18:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:18:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
2026-07-26 14:18:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:18:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-26 14:18:43 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-26 14:18:44 | INFO    

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

2026-07-26 14:20:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:20:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json "HTTP/1.1 200 OK"
Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'min_new_tokens', 'no_repeat_ngram_size', 'num_beams', 'pad_token_id', 'repetition_penalty', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
2026-07-26 14:20:16 | INFO     | da7ee7_ai_service | LLM ready: Qwen/Qwen2.5-7B-Instruct | quantization: 4-bit NF4 | generation: {'max_new_tokens': 300, 'min_new_tokens': 16, 'do_sample': False, 'num_beams': 1, 'repetition_penalty': 1.2

## 5. Embedding model + Whisper

In [9]:
# 5a. Embedding model for FAISS (multilingual — handles Arabic/English lecture content).
# -----------------------------------------------------------------------------
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    model_kwargs={"device": DEVICE},
)

# 5b. Whisper model for audio/video transcription.
whisper_model = whisper.load_model("medium", device=DEVICE)
logger.info("Embedding model and Whisper model loaded.")


2026-07-26 14:20:19 | INFO     | datasets | TensorFlow version 2.20.0 available.
2026-07-26 14:20:19 | INFO     | datasets | JAX version 0.7.2 available.
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json "HTTP/1.1 200 OK"
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

2026-07-26 14:20:27 | INFO     | sentence_transformers.base.model | Loading SentenceTransformer model from sentence-transformers/paraphrase-multilingual-mpnet-base-v2.
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:20:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multil

...

2026-07-26 14:20:35 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-07-26 14:20:35 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:20:35 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/special_tokens_map.json "HTTP/1.1 200 OK"
2026-07-26 14:20:35 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

2026-07-26 14:20:35 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-07-26 14:20:38 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-07-26 14:20:38 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 14:20:38 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-07-26 14:20:38 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-07-26 14:20:38 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
100%|██████████████████████████████████████| 1.42G/1.42G [00:08<00:00, 182MiB/s]
2026-07-26 14:21:00 | INFO     | da7ee7_ai_service | Embedding model and Whisper model loaded.


## 6. Document loaders — PDF, PPTX, DOCX, audio, video

In [10]:
# 6a. Loader functions for every supported file type.
# -----------------------------------------------------------------------------
def load_pdf(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n".join((page.extract_text() or "") for page in reader.pages)


def load_pptx(path: Path) -> str:
    prs = Presentation(str(path))
    chunks = []
    for slide in prs.slides:
        for shape in slide.shapes:
            if hasattr(shape, "text") and shape.text:
                chunks.append(shape.text)
    return "\n".join(chunks)


def load_docx(path: Path) -> str:
    doc = DocxDocument(str(path))
    return "\n".join(p.text for p in doc.paragraphs)


def extract_audio_from_video(path: Path) -> Path:
    audio_path = path.with_suffix(".wav")
    clip = VideoFileClip(str(path))
    clip.audio.write_audiofile(str(audio_path), logger=None)
    clip.close()
    return audio_path


def transcribe_audio(path: Path) -> str:
    result = whisper_model.transcribe(str(path))
    return result.get("text", "")


def clean_text(raw_text: str) -> str:
    text = re.sub(r"\s+", " ", raw_text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    return text.strip()


def load_any_file(path: Path) -> str:
    suffix = path.suffix.lower()
    logger.info(f"Loading '{path.name}' (type={suffix})")
    try:
        if suffix == ".pdf":
            raw = load_pdf(path)
        elif suffix in (".ppt", ".pptx"):
            raw = load_pptx(path)
        elif suffix in (".doc", ".docx"):
            raw = load_docx(path)
        elif suffix in (".mp3", ".wav", ".m4a"):
            raw = transcribe_audio(path)
        elif suffix in (".mp4", ".mov", ".mkv"):
            audio_path = extract_audio_from_video(path)
            raw = transcribe_audio(audio_path)
        else:
            logger.warning(f"Unsupported file type '{suffix}' for '{path.name}' — skipping.")
            raw = ""
    except Exception as exc:
        logger.exception(f"Failed to load/transcribe '{path.name}': {exc}")
        raw = ""

    cleaned = clean_text(raw)
    logger.info(f"Loaded '{path.name}': {len(cleaned)} characters extracted.")
    return cleaned


## 7. Chunking, embeddings, FAISS, and the RetrievalQA chain (per session)

In [11]:
# 7a. In-memory session registry: session_id -> {retriever, vectorstore, documents, file_texts, filenames}
# -----------------------------------------------------------------------------
SESSIONS: dict[str, dict[str, Any]] = {}

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def build_session_index(session_id: str, file_paths: list[Path]) -> dict[str, Any]:
    """
    Load every uploaded file, chunk it, and build a FAISS index for the
    session. Also keeps:
      - `documents`: the FULL list of chunks (used by Map-Reduce summarization
        instead of a 5-result similarity search, so the whole course is
        actually summarized).
      - `file_texts`: filename -> full raw text (used by solve_exam_file() to
        reliably recover an exam file's full text — the original code tried
        to rebuild this from mismatched raw_texts/filenames lists and by
        reaching into FAISS's private `docstore._dict`, which is why
        solved_exam.pdf sometimes silently failed to generate).
    """
    documents: list[Document] = []
    filenames: list[str] = []
    file_texts: dict[str, str] = {}

    for path in file_paths:
        text = load_any_file(path)
        if not text:
            logger.warning(f"No extractable text in '{path.name}' — excluded from index.")
            continue

        filenames.append(path.name)
        file_texts[path.name] = text

        chunks = text_splitter.split_text(text)
        for chunk in chunks:
            documents.append(Document(page_content=chunk, metadata={"source": path.name}))

    if not documents:
        logger.error(f"Ingest failed for session={session_id}: no extractable text in any uploaded file.")
        raise ValueError("No extractable text found in the uploaded files.")

    logger.info(f"Building FAISS index: {len(documents)} chunks from {len(filenames)} file(s) (session={session_id})")
    vectorstore = FAISS.from_documents(documents, embedding_model)
    vectorstore.save_local(str(INDEX_DIR / session_id))
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    SESSIONS[session_id] = {
        "retriever": retriever,
        "vectorstore": vectorstore,
        "filenames": filenames,
        "documents": documents,      # full chunk list -> Map-Reduce summarization
        "file_texts": file_texts,    # filename -> full text -> solve_exam_file()
    }
    logger.info(f"Session '{session_id}' indexed successfully ({len(documents)} chunks).")
    return SESSIONS[session_id]


def get_session(session_id: str) -> dict[str, Any]:
    if session_id not in SESSIONS:
        logger.error(f"Unknown session_id '{session_id}' requested.")
        raise KeyError(f"Unknown session_id '{session_id}'. Upload files first.")
    return SESSIONS[session_id]


## 8. Prompt templates + generation chains

In [12]:
# 8a. Prompt templates — rewritten for concise, bullet-point, hallucination-free,
# context-only output. A dedicated CHUNK_SUMMARY_PROMPT drives the Map step of
# Map-Reduce summarization; SMART_SUMMARY_PROMPT now drives the Reduce step.
# -----------------------------------------------------------------------------
CHUNK_SUMMARY_PROMPT = PromptTemplate(
    input_variables=["context"],
    template=(
        "You are Da7ee7-El-Dof3a, a precise academic teaching assistant.\n"
        "Summarize ONLY the key facts, definitions, formulas, and concepts in "
        "the text below. Use short bullet points. Do NOT repeat any point. "
        "Do NOT add information that is not in the text. Do NOT add an intro "
        "or closing sentence — bullets only.\n\n"
        "Text:\n---------\n{context}\n---------\n\nBullet-point summary:"
    ),
)

SMART_SUMMARY_PROMPT = PromptTemplate(
    input_variables=["context"],
    template=(
        "You are Da7ee7-El-Dof3a, a precise academic teaching assistant.\n"
        "Below are bullet-point summaries covering every part of a full lecture "
        "course, in no particular order. Merge them into ONE final Smart "
        "Summary that:\n"
        "- Covers every distinct concept exactly once (remove cross-part duplicates)\n"
        "- Is organized in a logical topic order, using bullet points and short headings\n"
        "- Highlights key terms, definitions, and formulas\n"
        "- Uses ONLY the information given below — never invent facts\n"
        "- Has no filler text and no closing remarks\n\n"
        "Partial summaries:\n---------\n{context}\n---------\n\nFinal Smart Summary:"
    ),
)

IMPORTANT_TOPICS_PROMPT = PromptTemplate(
    input_variables=["context"],
    template=(
        "You are Da7ee7-El-Dof3a. Using ONLY the material below, identify the "
        "most important exam topics, ranked by how heavily they are emphasized. "
        "Return between 5 and 12 short topic names.\n"
        "Return STRICT JSON ONLY — a single JSON array of strings, nothing else, "
        "no markdown, no explanation. Example: [\"Topic A\", \"Topic B\"]\n\n"
        "Material:\n---------\n{context}\n---------\n\nJSON array:"
    ),
)

# -----------------------------------------------------------------------------
# SOLVE_EXAM_PROMPT — rewritten to enforce short, exam-revision-style answers.
# This directly targets the "answers are too long" bug: the previous version
# invited "brief, numbered reasoning steps" with no hard length rule, which
# combined with max_new_tokens=300 reliably produced multi-paragraph answers.
# The prompt now hard-caps every answer at 2-5 lines, forbids intros/closings,
# prefers bullets, and gives the model a one-shot example (per spec) so it has
# a concrete target to imitate. The "[General knowledge]" escape hatch from
# the previous version is REPLACED with the exact required fallback sentence,
# so answers stay grounded in the uploaded material only.
# -----------------------------------------------------------------------------
SOLVE_EXAM_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are Da7ee7-El-Dof3a, answering a university exam question for a student.\n"
        "Follow these rules strictly:\n"
        "- Answer ONLY the question below.\n"
        "- Use ONLY the provided context — never invent facts, never hallucinate.\n"
        "- Keep the ENTIRE answer between 2 and 5 lines. Never exceed 5 lines.\n"
        "- Use simple, direct language suitable for exam revision.\n"
        "- Include only the key points needed for full marks. Do NOT repeat information.\n"
        "- Do NOT write an introduction or a conclusion — go straight to the answer.\n"
        "- Prefer short bullet points (each starting with '-') whenever appropriate.\n"
        "- Stop generating as soon as the answer is complete.\n"
        "- If the context does not contain the answer, respond with EXACTLY this "
        "sentence and nothing else: \"The answer was not found in the uploaded material.\"\n\n"
        "Example:\n"
        "Question: Explain Deadlock.\n"
        "Answer:\n"
        "- Deadlock is a situation where two or more processes wait indefinitely for resources held by each other.\n"
        "- It occurs when none of the processes can continue execution.\n"
        "- The four necessary conditions are Mutual Exclusion, Hold and Wait, No Preemption, and Circular Wait.\n\n"
        "Context:\n---------\n{context}\n---------\n\n"
        "Question:\n{question}\n\nAnswer:"
    ),
)

REVISION_NOTES_PROMPT = PromptTemplate(
    input_variables=["summary", "topics"],
    template=(
        "You are Da7ee7-El-Dof3a. Using ONLY the Smart Summary and Important "
        "Topics below, create a FINAL revision sheet, under 400 words, with "
        "exactly these three sections and nothing else:\n"
        "1. Must-Know (5-10 short bullets)\n"
        "2. Formulas / Definitions (bullets — omit this section if none apply)\n"
        "3. Common Mistakes (2-4 short bullets)\n"
        "No introduction, no closing remarks, no repeated bullets.\n\n"
        "Smart Summary:\n---------\n{summary}\n---------\n\n"
        "Important Topics:\n---------\n{topics}\n---------\n\n"
        "Final Revision Notes:"
    ),
)


# -----------------------------------------------------------------------------
# 8b. LLM call helpers.
# -----------------------------------------------------------------------------
def build_chat_prompt(user_content: str) -> str:
    """
    Wrap a raw instruction string using the model's chat template.

    Every candidate model (Qwen2.5-Instruct, Llama-3.2-Instruct, Gemma-3-it,
    Phi-4-mini-instruct, Mistral-Instruct) is instruction-tuned and expects
    its own special chat markup (<|im_start|>, [INST], etc.). The original
    code fed these models a bare completion-style string, which is the
    single biggest contributor to rambling / hallucinated / looping output —
    instruct models were never trained to free-associate from raw text the
    way base models are. Routing every prompt through apply_chat_template()
    fixes this at the source.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are Da7ee7-El-Dof3a, a precise, factual teaching assistant. "
                "Never repeat sentences or words. Never invent information that "
                "is not in the provided context."
            ),
        },
        {"role": "user", "content": user_content},
    ]
    try:
        return llm_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        # Fallback for any tokenizer without a chat template configured.
        return user_content


# -----------------------------------------------------------------------------
# EXAM_GENERATION_KWARGS — a separate, SHORTER generation profile for exam
# answers, layered on top of the global GENERATION_KWARGS (cell 16).
#
# Root cause of "answers are too long": every call site previously shared the
# same max_new_tokens=300 budget meant for full summaries. That is far more
# than 2-5 lines of text need, so the model reliably used the whole budget.
#   - max_new_tokens=160   -> hard ceiling well under what 5 lines requires,
#                             enforced independently of prompt-following.
#   - min_new_tokens=8     -> allow very short answers (e.g. the "not found"
#                             fallback sentence) without being padded out.
#   - repetition_penalty=1.3 / no_repeat_ngram_size=3 -> slightly stronger
#     than the summary profile, since short exam answers are exactly where a
#     repetition loop burns through the whole token budget on one repeated
#     bullet.
# do_sample/num_beams are inherited from GENERATION_KWARGS (greedy decoding).
# -----------------------------------------------------------------------------
EXAM_GENERATION_KWARGS = dict(
    GENERATION_KWARGS,
    max_new_tokens=160,
    min_new_tokens=8,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
)


def run_llm(prompt_text: str, **generation_overrides) -> str:
    """
    Run one generation call. Because the pipeline is configured with
    return_full_text=False (see cell 16), `generated_text` already excludes
    the prompt — no manual substring-stripping needed (the old
    `output[len(prompt_text):]` logic broke silently whenever the chat
    template changed the text length, leaking the raw prompt into results).

    `**generation_overrides` (e.g. EXAM_GENERATION_KWARGS) are passed
    straight through to the HF pipeline call, where they override the
    defaults baked in at pipeline-construction time — this is what lets
    exam answers use a much shorter max_new_tokens than summaries/revision
    notes without needing a second pipeline object.
    """
    chat_prompt = build_chat_prompt(prompt_text)
    result = text_gen_pipeline(chat_prompt, **generation_overrides)[0]["generated_text"]
    return result.strip()


def join_context(docs: list[Document], max_chars: int = 6000) -> str:
    joined = "\n\n".join(d.page_content for d in docs)
    return joined[:max_chars]


def batch_documents(documents: list[Document], max_chars: int = 3000) -> list[list[Document]]:
    """Group chunks into batches under `max_chars` so each Map-step LLM call
    gets a bounded, predictable context size."""
    batches: list[list[Document]] = []
    current: list[Document] = []
    current_len = 0
    for doc in documents:
        doc_len = len(doc.page_content)
        if current and current_len + doc_len > max_chars:
            batches.append(current)
            current, current_len = [], 0
        current.append(doc)
        current_len += doc_len
    if current:
        batches.append(current)
    return batches


def parse_json_topics(raw: str) -> list[str]:
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if not match:
        logger.warning("Important-topics output was not valid JSON — falling back to line-split parsing.")
        return [line.strip("- ").strip() for line in raw.splitlines() if line.strip()][:10]
    try:
        parsed = json.loads(match.group(0))
        return [str(t).strip() for t in parsed if str(t).strip()]
    except json.JSONDecodeError:
        logger.warning("Failed to json.loads() the matched topics array — falling back to line-split parsing.")
        return [line.strip("- ").strip() for line in raw.splitlines() if line.strip()][:10]


# -----------------------------------------------------------------------------
# Question splitting.
#
# FIX (root cause of "treats the entire exam as one single question"): the
# previous pattern only matched a numbering marker when it was *already*
# preceded by a literal newline in the extracted text, and only recognized
# Latin numbering styles (Q1, 1., 1), 1:, 1-). In practice:
#   - A marker at the very start of the extracted text (position 0) has no
#     leading "\n", so it never matched — harmless on its own, but any file
#     whose FIRST question also happened to be its ONLY cleanly-numbered
#     question would fall through to the "<=1 questions" case below.
#   - Exams using Arabic numbering/markers ("١.", "السؤال الأول", "س1:") never
#     matched at all, so the whole file came back as a single chunk.
#   - PDF/PPTX text extraction sometimes drops blank lines between questions
#     entirely, so even correctly Latin-numbered exams could fail to match.
# This version (a) prepends a sentinel newline so a marker at position 0 is
# still detected, (b) recognizes Arabic-Indic digits and common Arabic
# question markers in addition to the original Latin styles, and (c) falls
# back to blank-line / question-mark splitting if the numbered pattern still
# only finds one "question", instead of silently treating the whole exam as
# a single question.
# -----------------------------------------------------------------------------
QUESTION_SPLIT_PATTERN = re.compile(
    r"\n(?=\s*(?:"
    r"Q(?:uestion)?\s*\.?\s*\d+\s*[\.\):\-]?"                       # Q1, Question 1, Q.1
    r"|\d+\s*[\.\):\-]"                                             # 1.  1)  1:  1-
    r"|[\u0660-\u0669]+\s*[\.\):\-]"                                 # ١.  ٢)  (Arabic-Indic digits)
    r"|(?:ال)?سؤال\s*(?:رقم)?\s*[\d\u0660-\u0669]*\s*[:\-]?"        # سؤال / السؤال / سؤال رقم ١
    r"|س\s*[\d\u0660-\u0669]+\s*[:\-]"                               # س1:  (shorthand)
    r"))",
    re.IGNORECASE,
)

# Blank-line fallback, used only when numbered-marker splitting fails.
_BLANK_LINE_SPLIT = re.compile(r"\n\s*\n+")

# Sentence-level last resort: split after a Latin or Arabic question mark.
_QUESTION_MARK_SPLIT = re.compile(r"(?<=[?\u061F])\s+")


def split_questions(full_text: str) -> list[str]:
    """
    Split raw exam text into individual questions. Tries, in order:
      1. Numbered/lettered markers (Latin + Arabic styles).
      2. Blank-line-separated paragraphs.
      3. Sentences ending in '?' / '؟'.
    Returns an empty list only if NONE of these produce more than one usable
    fragment, so the caller can return a meaningful error instead of silently
    producing an empty solved_exam.pdf.
    """
    if not full_text or not full_text.strip():
        return []

    stripped = full_text.strip()
    # Sentinel leading newline so a marker sitting at position 0 (start of
    # the file/exam) still creates a split boundary via the lookahead above.
    normalized = "\n" + stripped

    raw_questions = QUESTION_SPLIT_PATTERN.split(normalized)
    questions = [q.strip() for q in raw_questions if len(q.strip()) > 10]

    if len(questions) <= 1:
        logger.warning(
            "split_questions: numbered-marker split found <=1 question — "
            "trying blank-line fallback split."
        )
        fallback = [q.strip() for q in _BLANK_LINE_SPLIT.split(stripped) if len(q.strip()) > 10]
        if len(fallback) > 1:
            questions = fallback
        else:
            logger.warning(
                "split_questions: blank-line fallback also found <=1 question — "
                "trying sentence-level ('?'/'؟') fallback split."
            )
            sentence_fallback = [
                q.strip() for q in _QUESTION_MARK_SPLIT.split(stripped) if len(q.strip()) > 10
            ]
            if len(sentence_fallback) > 1:
                questions = sentence_fallback

    questions = questions[:30]

    logger.info(f"split_questions: detected {len(questions)} question(s).")
    for i, q in enumerate(questions, start=1):
        preview = q[:120].replace("\n", " ")
        logger.info(f"  Q{i}: {preview}{'...' if len(q) > 120 else ''}")

    return questions


def _dedupe_repeated_lines(text: str) -> str:
    """
    Defensive cleanup for repetition loops: collapse consecutive duplicate
    lines/bullets. `no_repeat_ngram_size` blocks exact n-gram repeats within
    the model's own vocabulary window, but a short bullet (e.g. a 3-4 word
    line) can still legally repeat verbatim on the next line without
    tripping it — this catches that case as a second line of defense.
    """
    lines = [l.strip() for l in text.split("\n")]
    deduped: list[str] = []
    for line in lines:
        if line and deduped and line == deduped[-1]:
            continue
        deduped.append(line)
    return "\n".join(deduped)


def _truncate_answer(text: str, max_lines: int = 5) -> str:
    """
    Hard safety net for the '2-5 lines' requirement. The prompt asks the
    model to stop on its own, but instruction-following isn't guaranteed —
    this guarantees the constraint regardless of what the model generates.
    """
    lines = [l for l in text.split("\n") if l.strip()]
    if len(lines) > max_lines:
        lines = lines[:max_lines]
    return "\n".join(lines).strip()


In [13]:
# 8c. High-level generation functions used by the FastAPI endpoints.
# -----------------------------------------------------------------------------
def summarize_map_reduce(session_id: str) -> str:
    """
    Summarize the ENTIRE course, not just the top-5 chunks a similarity
    search happens to retrieve:
      1. MAP    — every chunk stored for this session is summarized in
                  batches (no FAISS search involved — we already have every
                  chunk from ingest, so we reuse it directly).
      2. REDUCE — the batch summaries are merged into one final,
                  de-duplicated Smart Summary.
    Results are cached on the session so generate_important_topics() can
    reuse them instead of re-running the whole Map step (avoids duplicate
    LLM calls).
    """
    session = get_session(session_id)
    documents = session["documents"]
    batches = batch_documents(documents, max_chars=3000)

    logger.info(f"Map-reduce summary: {len(documents)} chunks -> {len(batches)} batch(es) (session={session_id})")

    partial_summaries = []
    for i, batch in enumerate(batches, start=1):
        context = join_context(batch, max_chars=3000)
        partial = run_llm(CHUNK_SUMMARY_PROMPT.format(context=context))
        partial_summaries.append(partial)
        logger.info(f"Map step {i}/{len(batches)} complete (session={session_id})")

    merged_context = "\n\n".join(f"- {s}" for s in partial_summaries)
    logger.info(f"Reducing {len(partial_summaries)} partial summaries into final summary (session={session_id})")
    final_summary = run_llm(SMART_SUMMARY_PROMPT.format(context=merged_context))

    session["last_summary"] = final_summary
    session["last_partial_summaries"] = partial_summaries
    return final_summary


def generate_smart_summary(session_id: str) -> str:
    return summarize_map_reduce(session_id)


def generate_important_topics(session_id: str) -> list[str]:
    session = get_session(session_id)
    if "last_partial_summaries" not in session:
        # Make sure we have full-course coverage before extracting topics.
        summarize_map_reduce(session_id)
    context = "\n".join(session["last_partial_summaries"])
    logger.info(f"Generating important topics (session={session_id})")
    raw = run_llm(IMPORTANT_TOPICS_PROMPT.format(context=context))
    topics = parse_json_topics(raw)
    session["last_topics"] = topics
    return topics


def generate_revision_notes(session_id: str, summary: str, topics: list[str]) -> str:
    logger.info(f"Generating revision notes (session={session_id})")
    notes = run_llm(REVISION_NOTES_PROMPT.format(summary=summary, topics=", ".join(topics)))
    get_session(session_id)["last_revision_notes"] = notes
    return notes


def answer_question(session_id: str, question: str) -> dict[str, Any]:
    """
    Answer a single exam question against this session's retrieved context.

    FIX: this now (a) uses the shorter EXAM_GENERATION_KWARGS profile instead
    of the 300-token summary profile, directly addressing "answers are too
    long"; (b) runs the output through _dedupe_repeated_lines() +
    _truncate_answer() as a hard safety net for repetition loops / the 2-5
    line limit; and (c) NEVER raises — any generation failure for a single
    question is logged and turned into a graceful fallback answer instead of
    aborting the whole exam. That last point is also the fix for
    solved_exam.pdf sometimes being missing: previously, one bad question
    (a transient generation error, a retrieval edge case, etc.) raised out of
    the list comprehension in solve_exam_file(), aborting /solve-exam before
    export_pdf() was ever called — so NO pdf was produced even though most
    of the exam succeeded.
    """
    session = get_session(session_id)
    # `.get_relevant_documents()` is deprecated in current LangChain in favor
    # of `.invoke()`; using invoke() silences the deprecation warning.
    docs = session["retriever"].invoke(question)
    context = join_context(docs)
    if not context.strip():
        logger.warning(f"Empty retrieval context for a question (session={session_id})")
        context = "No relevant context found in the uploaded material."

    try:
        raw_answer = run_llm(
            SOLVE_EXAM_PROMPT.format(context=context, question=question),
            **EXAM_GENERATION_KWARGS,
        )
        answer = _truncate_answer(_dedupe_repeated_lines(raw_answer), max_lines=5)
        if not answer.strip():
            answer = "The answer was not found in the uploaded material."
        logger.info(
            f"answer_question: generated {len(answer)} char(s) / "
            f"{len(answer.splitlines())} line(s) (session={session_id})"
        )
    except Exception:
        logger.exception(
            f"answer_question: generation failed for question (session={session_id}): {question[:80]!r}"
        )
        answer = "The answer was not found in the uploaded material."

    return {"question": question.strip(), "answer": answer, "confidence": None}


def solve_exam_file(session_id: str, exam_filename: str) -> list[dict[str, Any]]:
    """
    FIX: the original implementation tried to rebuild an exam file's full
    text via `zip(session["raw_texts"], session["filenames"] * 1)` — but
    raw_texts is a list of CHUNKS while filenames is a list of FILES, so the
    lengths never matched and pairs were silently wrong. It then fell back to
    reaching into FAISS's private `vectorstore.docstore._dict`, which is
    fragile across LangChain versions. Both paths could return an empty or
    wrong `full_text`, which is why solve_exam sometimes produced an empty
    solved_exam.pdf or failed outright.

    Fix: `file_texts` (filename -> full raw text) is now stored directly on
    the session at ingest time (see build_session_index), so lookup here is
    a simple, reliable dict access.

    Also: each question is now solved via answer_question(), which never
    raises (see above) — so a problem with one question can no longer take
    down the whole request. GPU memory is freed every few questions, since a
    long exam (20-30+ questions) generating back-to-back was the other main
    trigger for a mid-request CUDA OOM that aborted /solve-exam before
    export_pdf() ran.
    """
    session = get_session(session_id)
    file_texts = session.get("file_texts", {})

    full_text = file_texts.get(exam_filename)
    if full_text is None:
        available = ", ".join(file_texts.keys()) or "none"
        logger.error(f"solve_exam_file: '{exam_filename}' not found in session={session_id} (available: {available})")
        raise KeyError(f"Exam file '{exam_filename}' was not found in this session. Uploaded files: {available}")

    questions = split_questions(full_text)
    if not questions:
        logger.warning(f"solve_exam_file: no questions detected in '{exam_filename}' (session={session_id})")
        raise ValueError(
            f"No questions could be detected in '{exam_filename}'. "
            "Supported numbering styles: '1.', '1)', 'Q1', 'Question 1', 'السؤال 1'."
        )

    logger.info(f"solve_exam_file: solving {len(questions)} question(s) from '{exam_filename}' (session={session_id})")

    solved: list[dict[str, Any]] = []
    for i, q in enumerate(questions, start=1):
        solved.append(answer_question(session_id, q))
        if DEVICE == "cuda" and i % 5 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    return solved


## 9. PDF export

In [14]:
# 9a. Export any generated text as a downloadable, formatted PDF via ReportLab.
# -----------------------------------------------------------------------------
def _pdf_styles():
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="Heading2Custom", parent=styles["Heading2"], spaceBefore=12, spaceAfter=6))
    styles.add(ParagraphStyle(name="BulletCustom", parent=styles["BodyText"], leftIndent=14, spaceAfter=4))
    return styles


def _is_heading(line: str) -> bool:
    stripped = line.strip().strip("*")
    if stripped.startswith("#"):
        return True
    if stripped.endswith(":") and len(stripped.split()) <= 8:
        return True
    if re.match(r"^\d+\.\s+[A-Z]", stripped) and len(stripped) < 60 and not stripped.endswith("."):
        return True
    return False


def _is_bullet(line: str) -> bool:
    return bool(re.match(r"^\s*[-\u2022*]\s+", line))


def export_pdf(session_id: str, file_type: str, content: str) -> Path:
    """
    Render generated text (summary / revision notes / solved exam) as a
    formatted PDF: title, headings, bullet lists, wrapped paragraphs, and
    page breaks for long content. Every text fragment is XML-escaped before
    being handed to ReportLab's Paragraph, which otherwise interprets raw
    '&', '<', '>' characters as markup and raises mid-build — this was a
    direct cause of missing solved_exam.pdf files whenever exam/lecture text
    contained those characters (e.g. "A < B", "R&D", HTML-like snippets).
    """
    out_dir = OUTPUT_DIR / session_id
    out_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = out_dir / f"{file_type}.pdf"

    logger.info(f"PDF export started: file_type='{file_type}' session={session_id} ({len(content or '')} chars) -> {pdf_path}")

    styles = _pdf_styles()
    story = [Paragraph(xml_escape(file_type.replace("_", " ").title()), styles["Title"]), Spacer(1, 16)]

    if not content or not content.strip():
        story.append(Paragraph("(No content was generated.)", styles["BodyText"]))
    else:
        bullet_buffer: list[str] = []

        def flush_bullets():
            if bullet_buffer:
                items = [ListItem(Paragraph(xml_escape(b), styles["BulletCustom"])) for b in bullet_buffer]
                story.append(ListFlowable(items, bulletType="bullet", start="circle"))
                bullet_buffer.clear()

        for raw_line in content.split("\n"):
            line = raw_line.strip()
            if not line:
                continue
            if line in ("\f", "<pagebreak>"):
                flush_bullets()
                story.append(PageBreak())
                continue
            if _is_bullet(line):
                bullet_buffer.append(re.sub(r"^\s*[-\u2022*]\s+", "", line))
                continue
            flush_bullets()
            if _is_heading(line):
                story.append(Paragraph(xml_escape(line.lstrip("#").strip()), styles["Heading2Custom"]))
            else:
                story.append(Paragraph(xml_escape(line), styles["BodyText"]))
                story.append(Spacer(1, 6))
        flush_bullets()

    try:
        doc = SimpleDocTemplate(
            str(pdf_path), pagesize=A4,
            leftMargin=2 * cm, rightMargin=2 * cm, topMargin=2 * cm, bottomMargin=2 * cm,
        )
        doc.build(story)
    except Exception as exc:
        logger.exception(f"PDF export failed: file_type='{file_type}' session={session_id}")
        raise RuntimeError(f"Failed to export PDF '{file_type}': {exc}") from exc

    exists = pdf_path.exists()
    logger.info(f"PDF export completed: file_type='{file_type}' session={session_id} path={pdf_path} exists={exists}")

    if not exists:
        logger.error(f"PDF export reported no error but file is missing: {pdf_path}")
        raise RuntimeError(f"PDF export reported success but file was not created: {pdf_path}")

    logger.info(f"Exported PDF '{pdf_path}' ({pdf_path.stat().st_size} bytes)")
    return pdf_path


## 10. FastAPI app exposed to the local backend (via ngrok)

In [15]:
# 10a. Define the FastAPI app + endpoints matching what backend/app/services/kaggle_client.py expects.
# -----------------------------------------------------------------------------
kaggle_app = FastAPI(title="Da7ee7-El-Dof3a AI Service")

ALLOWED_DOWNLOAD_TYPES = {"summary", "revision_notes", "solved_exam"}


def _error_response(status_code: int, message: str, **extra) -> JSONResponse:
    logger.error(f"API error ({status_code}): {message}")
    return JSONResponse(status_code=status_code, content={"status": "error", "detail": message, **extra})


@kaggle_app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": SELECTED_MODEL_NAME,
        "quantization": QUANT_MODE,
        "device": DEVICE,
        "active_sessions": len(SESSIONS),
    }


@kaggle_app.post("/ingest")
async def ingest(session_id: str = Form(...), files: list[UploadFile] = File(...)):
    if not files:
        return _error_response(400, "No files were provided.")

    session_dir = UPLOADS_DIR / session_id
    session_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Ingest requested: session={session_id}, {len(files)} file(s)")

    saved_paths = []
    try:
        for f in files:
            dest = session_dir / f.filename
            with dest.open("wb") as out:
                shutil.copyfileobj(f.file, out)
            saved_paths.append(dest)
    except Exception as exc:
        logger.exception(f"Failed saving uploaded files for session={session_id}")
        return _error_response(500, f"Failed to save uploaded files: {exc}")

    try:
        build_session_index(session_id, saved_paths)
    except ValueError as exc:
        return _error_response(400, str(exc))
    except Exception as exc:
        logger.exception(f"Failed to build FAISS index for session={session_id}")
        return _error_response(500, f"Failed to index uploaded files: {exc}")

    return {"session_id": session_id, "status": "indexed", "files": [p.name for p in saved_paths]}


@kaggle_app.post("/generate-summary")
async def generate_summary_endpoint(payload: dict):
    session_id = payload.get("session_id")
    if not session_id:
        return _error_response(400, "session_id is required.")

    try:
        get_session(session_id)
    except KeyError as exc:
        return _error_response(404, str(exc))

    try:
        summary = generate_smart_summary(session_id)
        topics = generate_important_topics(session_id)
        revision = generate_revision_notes(session_id, summary, topics)

        export_pdf(session_id, "summary", summary)
        export_pdf(session_id, "revision_notes", revision)
    except Exception as exc:
        logger.exception(f"generate-summary failed for session={session_id}")
        return _error_response(500, f"Failed to generate summary: {exc}")

    return {
        "session_id": session_id,
        "smart_summary": summary,
        "important_topics": topics,
        "revision_notes": revision,
        "pdf_url": f"/download?session_id={session_id}&file_type=summary",
        "revision_pdf_url": f"/download?session_id={session_id}&file_type=revision_notes",
    }


@kaggle_app.post("/solve-exam")
async def solve_exam_endpoint(payload: dict):
    session_id = payload.get("session_id")
    exam_filename = payload.get("exam_filename")
    question_text = payload.get("question_text")

    if not session_id:
        return _error_response(400, "session_id is required.")

    try:
        get_session(session_id)
    except KeyError as exc:
        return _error_response(404, str(exc))

    if not question_text and not exam_filename:
        return _error_response(400, "Provide 'exam_filename' or 'question_text'.")

    try:
        if question_text:
            logger.info(f"solve-exam: answering a single free-text question (session={session_id})")
            solved = [answer_question(session_id, question_text)]
        else:
            solved = solve_exam_file(session_id, exam_filename)
    except KeyError as exc:
        return _error_response(404, str(exc))
    except ValueError as exc:
        # e.g. "no questions detected in this file"
        return _error_response(422, str(exc))
    except Exception as exc:
        logger.exception(f"solve-exam failed for session={session_id}")
        return _error_response(500, f"Failed to solve exam: {exc}")

    logger.info(f"solve-exam: {len(solved)} question(s) answered (session={session_id})")

    try:
        combined_text = "\n\n".join(f"Q{i + 1}. {q['question']}\n{q['answer']}" for i, q in enumerate(solved))
        pdf_path = export_pdf(session_id, "solved_exam", combined_text)
        logger.info(
            f"solve-exam: solved_exam.pdf ready for session={session_id} "
            f"at {pdf_path} (exists={pdf_path.exists()})"
        )
    except Exception as exc:
        logger.exception(f"Failed to export solved_exam.pdf for session={session_id}")
        return _error_response(500, f"Answers were generated but PDF export failed: {exc}")

    return {
        "session_id": session_id,
        "solved_questions": solved,
        "pdf_url": f"/download?session_id={session_id}&file_type=solved_exam",
    }


@kaggle_app.get("/download")
async def download_endpoint(session_id: str, file_type: str):
    if file_type not in ALLOWED_DOWNLOAD_TYPES:
        return _error_response(400, f"Invalid file_type '{file_type}'. Must be one of: {sorted(ALLOWED_DOWNLOAD_TYPES)}")

    pdf_path = OUTPUT_DIR / session_id / f"{file_type}.pdf"
    if not pdf_path.exists():
        logger.warning(f"Download requested but missing: session={session_id} file_type={file_type}")
        return _error_response(
            404,
            f"'{file_type}.pdf' has not been generated yet for this session. "
            f"Call /generate-summary or /solve-exam first.",
        )

    logger.info(f"Serving download: session={session_id} file_type={file_type}")
    return FileResponse(str(pdf_path), media_type="application/pdf", filename=f"{file_type}.pdf")


@kaggle_app.exception_handler(Exception)
async def unhandled_exception_handler(request: Request, exc: Exception):
    # Safety net: guarantees the API NEVER returns a bare, undiagnosable
    # HTTP 500 — every failure reaches the client as structured JSON.
    logger.exception(f"Unhandled exception on {request.method} {request.url.path}")
    return JSONResponse(status_code=500, content={"status": "error", "detail": f"Internal server error: {exc}"})


## 11. ngrok tunnel — expose the Kaggle FastAPI server publicly

In [16]:
# 11a. Authenticate ngrok using a Kaggle secret / environment variable, then open the tunnel.
# -----------------------------------------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
except Exception:
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "")

if not NGROK_AUTH_TOKEN:
    raise RuntimeError(
        "NGROK_AUTH_TOKEN not found. Add it as a Kaggle secret or environment variable."
    )

ngrok_conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.kill()  # ensure no stale tunnels
public_tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = public_tunnel.public_url

print("=" * 60)
print(f"Da7ee7-El-Dof3a AI Service is live at: {PUBLIC_URL}")
print("Paste this URL into backend/.env as KAGGLE_AI_BASE_URL")
print("=" * 60)

2026-07-26 14:21:01 | INFO     | pyngrok.ngrok | Opening tunnel named: http-8000-3bba15d5-d3ae-43ef-bf47-36b879ddf636


2026-07-26 14:21:02 | INFO     | pyngrok.process | Overriding default auth token
2026-07-26 14:21:02 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:21:02+0000 lvl=info msg="no configuration paths supplied"
2026-07-26 14:21:02 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:21:02+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
2026-07-26 14:21:02 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:21:02+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=<nil>
2026-07-26 14:21:02 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:21:02+0000 lvl=info msg="FIPS 140 mode" enabled=false
2026-07-26 14:21:02 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:21:02+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
2026-07-26 14:21:02 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:21:02+0000 lvl=info msg="client session established" obj=tunnels.session
2026-07-26 14:

Da7ee7-El-Dof3a AI Service is live at: https://shudder-reviver-undercoat.ngrok-free.dev
Paste this URL into backend/.env as KAGGLE_AI_BASE_URL


2026-07-26 14:21:02 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:21:02+0000 lvl=info msg=end pg=/api/tunnels id=41a0e8a1b628c763 status=201 dur=171.30839ms


In [17]:
# 11b. Run the FastAPI server in a background thread so the notebook stays interactive.
# -----------------------------------------------------------------------------
def run_server():
    uvicorn.run(kaggle_app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)
logger.info(f"FastAPI server running locally on :8000 and publicly at {PUBLIC_URL}")
logger.info("Keep this notebook session running while the backend is in use.")


INFO:     Started server process [167]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
2026-07-26 14:21:05 | INFO     | da7ee7_ai_service | FastAPI server running locally on :8000 and publicly at https://shudder-reviver-undercoat.ngrok-free.dev
2026-07-26 14:21:05 | INFO     | da7ee7_ai_service | Keep this notebook session running while the backend is in use.
2026-07-26 14:27:40 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:27:40+0000 lvl=info msg="join connections" obj=join id=f232a9454578 l=127.0.0.1:8000 r=156.221.136.5:11642
2026-07-26 14:27:41 | INFO     | da7ee7_ai_service | Ingest requested: session=8f8fcf52f80e, 4 file(s)
2026-07-26 14:27:41 | INFO     | da7ee7_ai_service | Loading 'Operating_Systems_Exam_2022.pdf' (type=.pdf)
2026-07-26 14:27:41 | INFO     | da7ee7_ai_service | Loaded 'Operating_Systems_Exam_2022.pdf': 323 characters extracted.
2026-07-26 14:27:41 

INFO:     156.221.136.5:0 - "POST /ingest HTTP/1.1" 200 OK


2026-07-26 14:27:44 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:27:44+0000 lvl=info msg="join connections" obj=join id=e66a928893e1 l=127.0.0.1:8000 r=156.221.136.5:11646
2026-07-26 14:27:44 | INFO     | da7ee7_ai_service | Map-reduce summary: 5 chunks -> 1 batch(es) (session=8f8fcf52f80e)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=16) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-07-26 14:27:54 | INFO     | da7ee7_ai_service | Map step 1/1 complete (session=8f8fcf52f80e)
2026-07-26 14:27:54 | INFO     | da7ee7_ai_service | Reducing 1 partial summaries into final summary (sess

INFO:     156.221.136.5:0 - "POST /generate-summary HTTP/1.1" 200 OK


2026-07-26 14:28:41 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:28:41+0000 lvl=info msg="join connections" obj=join id=df0aea1457a4 l=127.0.0.1:8000 r=156.221.136.5:11674
2026-07-26 14:28:41 | INFO     | da7ee7_ai_service | Serving download: session=8f8fcf52f80e file_type=summary


...

2026-07-26 14:43:16 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:43:16+0000 lvl=info msg="join connections" obj=join id=fe98ba71de7a l=127.0.0.1:8000 r=156.221.136.5:14347
2026-07-26 14:43:16 | INFO     | da7ee7_ai_service | Map-reduce summary: 2 chunks -> 1 batch(es) (session=791f5728ce65)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=16) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-07-26 14:43:29 | INFO     | da7ee7_ai_service | Map step 1/1 complete (session=791f5728ce65)
2026-07-26 14:43:29 | INFO     | da7ee7_ai_service | Reducing 1 partial summaries into final summary (sess

INFO:     156.221.136.5:0 - "POST /generate-summary HTTP/1.1" 200 OK


2026-07-26 14:46:16 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:46:16+0000 lvl=info msg="join connections" obj=join id=c015bf811c81 l=127.0.0.1:8000 r=156.221.136.5:12584
2026-07-26 14:46:16 | WARNING  | da7ee7_ai_service | split_questions: numbered-marker split found <=1 question — trying blank-line fallback split.
2026-07-26 14:46:16 | WARNING  | da7ee7_ai_service | split_questions: blank-line fallback also found <=1 question — trying sentence-level ('?'/'؟') fallback split.
2026-07-26 14:46:16 | INFO     | da7ee7_ai_service | split_questions: detected 3 question(s).
2026-07-26 14:46:16 | INFO     | da7ee7_ai_service |   Q1: Operating Systems Practice Exam 1. Explain Deadlock. 2. Compare Paging and Segmentation. 3. What is Virtual Memory?
2026-07-26 14:46:16 | INFO     | da7ee7_ai_service |   Q2: 4. Explain FCFS Scheduling. 5. Compare Processes and Threads. 6. Describe the Process Life Cycle. 7. Explain File Alloca...
2026-07-26 14:46:16 | INFO     | da7ee7_ai_service |   Q3

INFO:     156.221.136.5:0 - "POST /solve-exam HTTP/1.1" 200 OK


2026-07-26 14:48:29 | INFO     | pyngrok.process.ngrok | t=2026-07-26T14:48:29+0000 lvl=info msg="join connections" obj=join id=d0efc63feef0 l=127.0.0.1:8000 r=156.221.136.5:14479


## Notes

- **Gated models:** Llama and Gemma checkpoints require accepting their
  license on Hugging Face and providing an `HF_TOKEN` secret. If a model
  fails to load, the comparison step skips it and continues with the rest.
- **Session persistence:** `SESSIONS` and the FAISS indexes live in this
  notebook's memory/disk for the duration of the Kaggle session. Restarting
  the notebook clears them — students would need to re-upload.
- **Re-running after a Kaggle restart:** ngrok assigns a new public URL each
  time. Update `KAGGLE_AI_BASE_URL` in the backend's `.env` file accordingly.